# 💾 Notebook 08 – Model Saving and Testing
**Goal:** export the final model so the Streamlit app can use it, and test everything the way the app will.

**Steps**
1. Load the optimized pipeline and split it into `sentiment_model.pkl` + `tfidf_vectorizer.pkl`
2. Write `04_App/preprocessing.py` (same cleaning code as in training)
3. Test single predictions, edge cases and batch predictions
4. Run final checks on the project files

## 1. Import Libraries

In [ ]:
# Import libraries
import sys
import json
import time
import warnings
import importlib
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 100)

In [ ]:
# Set up project paths (works on your D: drive path, or falls back to the parent of 02_Notebooks)
from pathlib import Path

BASE_DIR = Path(r"D:\DataScience\notebooks\SentimentIQ")
if not BASE_DIR.exists():
    BASE_DIR = Path.cwd().parent          # fallback: notebook is running inside 02_Notebooks/

RAW_PATH       = BASE_DIR / "01_Data" / "Raw" / "Tweets.csv"
PROCESSED_DIR  = BASE_DIR / "01_Data" / "Processed"
CLEANED_PATH   = PROCESSED_DIR / "cleaned_tweets.csv"
MODELS_DIR     = BASE_DIR / "03_Models"
REPORTS_DIR    = BASE_DIR / "05_Reports"

for folder in (PROCESSED_DIR, MODELS_DIR, REPORTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42      # same seed everywhere -> reproducible results
TEST_SIZE    = 0.20    # 80% training / 20% testing

print("Project folder :", BASE_DIR)
print("Raw data exists:", RAW_PATH.exists())

In [ ]:
# Load the cleaned dataset and recreate the SAME train/test split that was made in Notebook 4
import pandas as pd
from sklearn.model_selection import train_test_split

try:
    df = pd.read_csv(CLEANED_PATH)
except FileNotFoundError:
    raise FileNotFoundError("cleaned_tweets.csv not found. Please run Notebook 03 first.")

df = df.dropna(subset=["cleaned_text", "airline_sentiment"]).reset_index(drop=True)

X = df["cleaned_text"]          # input feature
y = df["airline_sentiment"]     # target variable

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print("Train size:", len(X_train), "| Test size:", len(X_test))

## 2. Load the Optimized Pipeline

In [ ]:
# Load the pipeline saved in Notebook 07
try:
    pipeline = joblib.load(MODELS_DIR / "optimized_pipeline.pkl")
except FileNotFoundError:
    raise FileNotFoundError("optimized_pipeline.pkl not found. Please run Notebook 07 first.")

print("Pipeline steps:", [name for name, _ in pipeline.steps])
print("Classifier    :", type(pipeline.named_steps["clf"]).__name__)
print("Vocabulary    :", len(pipeline.named_steps["tfidf"].vocabulary_), "terms")

## 3. Save the Final Model and Vectorizer Separately

In [ ]:
# The app needs two files: the vectorizer (text -> numbers) and the model (numbers -> sentiment)
vectorizer = pipeline.named_steps["tfidf"]
model      = pipeline.named_steps["clf"]

try:
    joblib.dump(model,      MODELS_DIR / "sentiment_model.pkl")
    joblib.dump(vectorizer, MODELS_DIR / "tfidf_vectorizer.pkl")     # replaces the Notebook 04 vectorizer
    print("✅ sentiment_model.pkl and tfidf_vectorizer.pkl saved in", MODELS_DIR)
except Exception as e:
    print("❌ Saving failed:", e)

> The vectorizer saved here **overwrites** the one from Notebook 04 on purpose: the final model was trained with these tuned TF-IDF settings, and a model only works with its own vectorizer.

In [ ]:
# Save metadata (useful for the app's "About" page and for reproducibility)
final_pred = pipeline.predict(X_test)
metadata = {
    "project": "SentimentIQ",
    "model_type": type(model).__name__,
    "classes": list(model.classes_),
    "vocabulary_size": len(vectorizer.vocabulary_),
    "ngram_range": list(vectorizer.ngram_range),
    "test_accuracy": round(float(accuracy_score(y_test, final_pred)), 4),
    "test_macro_f1": round(float(f1_score(y_test, final_pred, average="macro")), 4),
    "train_samples": int(len(X_train)),
    "test_samples": int(len(X_test)),
    "sklearn_version": sklearn.__version__,
}
with open(MODELS_DIR / "model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print(json.dumps(metadata, indent=2))

## 4. Write `04_App/preprocessing.py`
The app must clean text **exactly** like the training data. This cell writes the same preprocessing code used in Notebook 03 into the app folder.

In [ ]:
# Write the shared preprocessing module into the app folder
PREPROCESSING_SOURCE = r'''
# preprocessing.py - text cleaning shared by the notebooks and the Streamlit app
import html
import re

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import RegexpTokenizer

# Download the NLTK resources once (quietly skipped if already installed / offline)
for _resource in ("stopwords", "wordnet", "omw-1.4"):
    try:
        nltk.download(_resource, quiet=True)
    except Exception:
        pass

# Words that can flip the meaning of a sentence must NOT be removed
NEGATION_WORDS = {"no", "nor", "not", "never", "neither", "none",
                  "nothing", "nobody", "nowhere", "cannot", "without"}
KEEP_WORDS = NEGATION_WORDS | {"very", "too", "but"}   # negations + a few strong modifiers

STOP_WORDS = set(stopwords.words("english")) - KEEP_WORDS

# Expand contractions so that "don't" becomes "do not" (keeps the negation visible)
CONTRACTIONS = [
    (r"won't", "will not"), (r"can't", "can not"), (r"cannot", "can not"),
    (r"shan't", "shall not"), (r"n't", " not"), (r"'re", " are"),
    (r"'ve", " have"), (r"'ll", " will"), (r"'d", " would"), (r"'m", " am"),
]

_tokenizer = RegexpTokenizer(r"[a-z]+")
_lemmatizer = WordNetLemmatizer()


def normalize_text(text):
    # Lowercase, strip HTML/URLs/mentions/hashtag symbols, expand contractions, keep letters only
    text = html.unescape(str(text)).lower()
    text = text.replace("\u2019", "'").replace("\u2018", "'")      # curly apostrophes
    text = re.sub(r"</?[a-z][^>]*>", " ", text)                    # HTML tags
    text = re.sub(r"(https?://\S+|www\.\S+)", " ", text)           # URLs
    text = re.sub(r"@\w+", " ", text)                              # @mentions (airline handles / usernames)
    text = text.replace("#", " ")                                  # keep hashtag word, drop the symbol
    for pattern, replacement in CONTRACTIONS:
        text = re.sub(pattern, replacement, text)
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)                     # sooooo -> soo
    text = re.sub(r"[^a-z\s]", " ", text)                          # punctuation, digits, emojis
    return re.sub(r"\s+", " ", text).strip()


def tokenize(text):
    return _tokenizer.tokenize(text)


def remove_stopwords(tokens):
    return [t for t in tokens if t not in STOP_WORDS and len(t) > 1]


def lemmatize_tokens(tokens):
    lemmas = []
    for token in tokens:
        lemma = _lemmatizer.lemmatize(token, pos="v")   # delayed -> delay
        lemma = _lemmatizer.lemmatize(lemma, pos="n")   # flights -> flight
        lemmas.append(lemma)
    return lemmas


def clean_text(text):
    # Full pipeline. Returns "" when the input is empty or has no useful words left.
    if not isinstance(text, str) or not text.strip():
        return ""
    tokens = tokenize(normalize_text(text))
    tokens = remove_stopwords(tokens)
    tokens = lemmatize_tokens(tokens)
    return " ".join(tokens)

'''

APP_DIR = BASE_DIR / "04_App"
APP_DIR.mkdir(exist_ok=True)
(APP_DIR / "preprocessing.py").write_text(PREPROCESSING_SOURCE.lstrip(), encoding="utf-8")
print("✅ Written:", APP_DIR / "preprocessing.py")

In [ ]:
# Import the module exactly the way the Streamlit app will
sys.path.insert(0, str(APP_DIR))
import preprocessing
importlib.reload(preprocessing)
from preprocessing import clean_text

print(clean_text("@united I DON'T like the delay!!! https://t.co/xyz"))

## 5. Reload Test – Files on Disk Must Behave Like the Pipeline

In [ ]:
# Load the saved files from disk as a fresh app would, and compare predictions with the in-memory pipeline
loaded_model      = joblib.load(MODELS_DIR / "sentiment_model.pkl")
loaded_vectorizer = joblib.load(MODELS_DIR / "tfidf_vectorizer.pkl")

disk_pred = loaded_model.predict(loaded_vectorizer.transform(X_test))
assert np.array_equal(disk_pred, pipeline.predict(X_test)), "Saved files do not match the pipeline!"
print("✅ Saved model + vectorizer give identical predictions to the pipeline.")
print("Test accuracy :", round(accuracy_score(y_test, disk_pred), 4))
print("Test macro F1 :", round(f1_score(y_test, disk_pred, average="macro"), 4))

## 6. Build the Prediction Function Used by the App

In [ ]:
# One reusable function: raw text in -> label, confidence and class probabilities out
def predict_sentiment(text, model=loaded_model, vectorizer=loaded_vectorizer):
    if not isinstance(text, str) or not text.strip():
        return {"text": text, "label": None, "confidence": None,
                "message": "Empty input - please enter some text."}

    cleaned = clean_text(text)
    if not cleaned:
        return {"text": text, "cleaned_text": "", "label": None, "confidence": None,
                "message": "No meaningful words left after cleaning (only links, mentions, emojis or stop words)."}

    probs = model.predict_proba(vectorizer.transform([cleaned]))[0]
    best = int(probs.argmax())
    return {
        "text": text,
        "cleaned_text": cleaned,
        "label": model.classes_[best],
        "confidence": round(float(probs[best]), 4),
        "probabilities": {cls: round(float(p), 4) for cls, p in zip(model.classes_, probs)},
        "message": "ok",
    }

predict_sentiment("The crew was fantastic and the flight was on time!")

## 7. Test with Realistic Sentences

In [ ]:
# Positive / negative / neutral / negation examples
test_sentences = [
    ("Amazing service, the crew was so friendly. Thank you!",       "positive"),
    ("Best airline experience I have had in years",                  "positive"),
    ("Flight delayed 4 hours and nobody told us anything. Awful.",   "negative"),
    ("Lost my luggage and customer service is useless",              "negative"),
    ("What is the baggage allowance for international flights?",     "neutral"),
    ("Can I change my booking to next Tuesday?",                     "neutral"),
    ("The food was not good",                                        "negative"),
    ("The flight was not bad at all",                                "positive?"),   # tricky negation
    ("I will never fly with you again",                              "negative"),
]

rows = []
for text, expected in test_sentences:
    r = predict_sentiment(text)
    rows.append({"Text": text, "Expected": expected, "Predicted": r["label"], "Confidence": r["confidence"]})
pd.DataFrame(rows)

The *"not bad at all"* sentence is intentionally tricky – a bag-of-words model may get it wrong. Mention such limitations honestly in your README.

## 8. Test Edge Cases (the app must never crash)

In [ ]:
# Edge cases: empty text, whitespace, None, number, only URL, only emoji, very long text, other language
edge_cases = [
    "", "     ", None, 12345,
    "https://t.co/abc123",
    "@united @delta",
    "😡😡😡",
    "great " * 500,
    "Muy mal servicio, nunca más",
    "!!!???",
]

for case in edge_cases:
    try:
        r = predict_sentiment(case)
        shown = repr(case)[:30]
        print(f"{shown:<34} -> {str(r['label']):<9} | {r['message'][:60]}")
    except Exception as e:
        print(f"❌ {repr(case)[:30]} crashed: {e}")

## 9. Batch Prediction Test

In [ ]:
# Predict a batch of 15 real test tweets the way the app's CSV upload feature will
sample = df.loc[X_test.index].sample(15, random_state=RANDOM_STATE)[["text", "airline_sentiment"]].reset_index(drop=True)
results = [predict_sentiment(t) for t in sample["text"]]

sample["predicted"]  = [r["label"] for r in results]
sample["confidence"] = [r["confidence"] for r in results]
sample["correct"]    = sample["predicted"] == sample["airline_sentiment"]
display(sample)
print(f"Correct: {sample['correct'].sum()} / {len(sample)}")

In [ ]:
# Speed test: how long do 1,000 predictions take? (the app should feel instant)
texts = df.loc[X_test.index, "text"].head(1000).tolist()
start = time.time()
_ = [predict_sentiment(t) for t in texts]
elapsed = time.time() - start
print(f"1,000 predictions in {elapsed:.2f}s  ->  {elapsed:.1f} ms per tweet")

## 10. Final Project Checklist

In [ ]:
# Confirm that every file the app needs exists
required = [
    RAW_PATH,
    CLEANED_PATH,
    MODELS_DIR / "sentiment_model.pkl",
    MODELS_DIR / "tfidf_vectorizer.pkl",
    MODELS_DIR / "model_metadata.json",
    BASE_DIR / "04_App" / "preprocessing.py",
]
for path in required:
    status = "✅" if path.exists() else "❌ MISSING"
    print(f"{status}  {path.relative_to(BASE_DIR)}")

print("\nLibrary versions to pin in requirements.txt:")
print(f"scikit-learn=={sklearn.__version__}")
print(f"pandas=={pd.__version__}")
print(f"numpy=={np.__version__}")
print(f"joblib=={joblib.__version__}")

## ✅ Summary
- Final model → `03_Models/sentiment_model.pkl`, vectorizer → `03_Models/tfidf_vectorizer.pkl`, metadata → `model_metadata.json`.
- `04_App/preprocessing.py` was generated from the same code used for training (no train/serve mismatch).
- `predict_sentiment()` handles empty text, URLs-only text, emojis, numbers and very long input without crashing.

**Next:** build the Streamlit app (`app.py`, `database.py`, `visualization.py`) in `04_App/`.
Use the pinned library versions printed above in `requirements.txt` – loading a `.pkl` with a different scikit-learn version can fail.